# ONG v3 - Open-set test: leave-K-genera-out RETRAIN (strict, deployment-realistic)

Notebook 21 (fixed deployed model) gave mean AUROC **0.963** but is optimistic - the backbone had
seen every genus. This notebook does the **strict** test for paper section 3.5(ii): it **removes K
genera from training entirely**, retrains DINOv2 on the rest, then asks whether those genuinely
**unseen** genera can be flagged as novel by embedding distance.

**One Run-all does everything** - modify splits -> retrain -> embed full set -> open-set AUROC +
figure. Outputs saved to the subfolder `MyDrive/ONG_v3/openset/` (dibuat otomatis - semua tetap di
dalam v3, tidak menyentuh data/model pilot asli).

Held-out (UNSEEN) genera: **Apostasia, Chrysoglossum, Spiranthes, Aglossorrhyncha, Dryadorchis, Vanda, Pterostylis, Paphiopedilum, Thrixspermum, Crepidium, Robiquetia, Oberonia**

### Before running
Use an **L4** GPU (Runtime -> Change runtime type -> L4), and make sure these are in
`MyDrive/ONG_v3/` (same files as the pilots): `scripts/03_train_bakeoff_colab.py`,
`data/splits/{{train,val,test}}_live.csv`, `photos.zip`. Then **Run all** (~15-20 compute units).


In [ ]:
# Sel 1 - Mount Drive, install deps, cek GPU
from google.colab import drive; drive.mount('/content/drive')
!pip -q install -U timm faiss-cpu scikit-learn
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert torch.cuda.is_available(), 'No GPU - Runtime > Change runtime type > L4, lalu Run all lagi.'

In [ ]:
# Sel 2 - Paths + daftar genus held-out, verifikasi file
import os
ROOT    = '/content/drive/MyDrive/ONG_v3'   # data asli (scripts, splits penuh, photos.zip)
OS_ROOT = f'{ROOT}/openset'                 # subfolder di dalam v3 (dibuat otomatis): splits
                                            # termodifikasi + model hold-out + hasil open-set
MODEL, IMG = 'dinov2l', 448
EVAL_NAME  = 'vit_large_patch14_reg4_dinov2.lvd142m'
HELD_OUT   = ['Apostasia', 'Chrysoglossum', 'Spiranthes', 'Aglossorrhyncha', 'Dryadorchis', 'Vanda', 'Pterostylis', 'Paphiopedilum', 'Thrixspermum', 'Crepidium', 'Robiquetia', 'Oberonia']

os.makedirs(f'{OS_ROOT}/data/splits', exist_ok=True)
need = ['scripts/03_train_bakeoff_colab.py', 'data/splits/train_live.csv',
        'data/splits/val_live.csv', 'data/splits/test_live.csv', 'photos.zip']
status = {f: os.path.exists(f'{ROOT}/{f}') for f in need}
print('present:', status)
missing = [f for f, ok in status.items() if not ok]
assert not missing, f'Missing di {ROOT}/: {missing} - upload dulu, lalu Run all lagi.'
print(f'{len(HELD_OUT)} genus akan di-hold-out:', HELD_OUT)

In [ ]:
# Sel 3 - Buat split termodifikasi (buang genus held-out dari train+val)
import pandas as pd
for name in ['train_live.csv', 'val_live.csv']:
    df = pd.read_csv(f'{ROOT}/data/splits/{name}')
    kept = df[~df['genus'].isin(HELD_OUT)].reset_index(drop=True)
    kept.to_csv(f'{OS_ROOT}/data/splits/{name}', index=False)
    print(f'{name}: {len(df):,} -> {len(kept):,} baris ({df["genus"].nunique()} -> '
          f'{kept["genus"].nunique()} genus)')

In [ ]:
# Sel 4 - Unzip photos ke disk lokal (/content/photos/{Genus}/*.jpg)
!unzip -q -o "{ROOT}/photos.zip" -d /content/
nested = '/content/photos/photos'
if os.path.isdir(nested):
    import shutil
    for it in os.listdir(nested):
        shutil.move(os.path.join(nested, it), '/content/photos')
    os.rmdir(nested)
assert os.path.isdir('/content/photos'), 'Tidak ada /content/photos setelah unzip.'
print('folder genus:', len(os.listdir('/content/photos')))

In [ ]:
# Sel 5 - Retrain DINOv2 pada split termodifikasi (genus held-out TIDAK pernah dilihat)
# Recipe identik dgn pilot: --sampler-power 0 --loss ce. Output -> {OS_ROOT}/models/dinov2l/
!python {ROOT}/scripts/03_train_bakeoff_colab.py --model {MODEL} --drive-root {OS_ROOT} \
    --photos-root /content/photos --img-size {IMG} --sampler-power 0 --loss ce \
    --warmup-epochs 3 --finetune-epochs 22 --seed 42
ckpt = f'{OS_ROOT}/models/{MODEL}/best_model_global.pth'
assert os.path.exists(ckpt), f'Checkpoint hilang: {ckpt} - training gagal (lihat log).'
print('OK ->', ckpt)

In [ ]:
# Sel 6 - Embed SELURUH gambar (cepat: fp16 autocast + DataLoader paralel + progress nyata)
# Aman dilanjut setelah disconnect: jalankan Sel 1,2,4 dulu (LEWATI Sel 5), lalu sel ini.
import json, time, numpy as np, torch, timm, pandas as pd
from torchvision import transforms
from PIL import Image
from torch.utils.data import Dataset, DataLoader

ckpt = f'{OS_ROOT}/models/{MODEL}/best_model_global.pth'   # checkpoint dari Sel 5 (di Drive)
assert os.path.exists(ckpt), f'Checkpoint hilang: {ckpt} - jalankan Sel 5 dulu.'
vocab = json.load(open(f'{OS_ROOT}/models/{MODEL}/vocab.json'))
print('genus dilatih:', len(vocab), flush=True)
m = timm.create_model(EVAL_NAME, pretrained=False, num_classes=len(vocab), img_size=IMG)
sd = torch.load(ckpt, map_location='cpu', weights_only=False)
m.load_state_dict(sd.get('state_dict', sd) if isinstance(sd, dict) else sd, strict=False)
cfg = timm.data.resolve_model_data_config(m); m = m.cuda().eval()
tfm = transforms.Compose([transforms.Resize(int(IMG*1.1)), transforms.CenterCrop(IMG),
                          transforms.ToTensor(), transforms.Normalize(cfg['mean'], cfg['std'])])

df = pd.concat([pd.read_csv(f'{ROOT}/data/splits/{s}_live.csv') for s in ('train','val','test')],
               ignore_index=True)
df['lp'] = df['path'].apply(lambda p: '/content/photos/' +
                            '/'.join(p.replace(chr(92), '/').split('/')[-2:]))

class _DS(Dataset):
    def __init__(self, paths): self.paths = paths
    def __len__(self): return len(self.paths)
    def __getitem__(self, i):
        try: return tfm(Image.open(self.paths[i]).convert('RGB'))
        except Exception: return torch.zeros(3, IMG, IMG)

dl = DataLoader(_DS(df['lp'].tolist()), batch_size=64, num_workers=4, pin_memory=True)
embs, done, t0 = [], 0, time.time()      # shuffle=False -> urutan = df, jadi meta tetap sejajar
with torch.no_grad(), torch.autocast('cuda', dtype=torch.float16):
    for x in dl:
        x = x.cuda(non_blocking=True)
        e = m.forward_head(m.forward_features(x), pre_logits=True)
        e = torch.nn.functional.normalize(e.float(), dim=1).cpu().numpy().astype('float32')
        embs.append(e); done += len(x); el = time.time() - t0
        print(f'  {done}/{len(df)} ({100*done/len(df):.0f}%)  {done/el:.0f} img/s  '
              f'elapsed {el:.0f}s  eta {(len(df)-done)/(done/el):.0f}s', flush=True)
emb = np.concatenate(embs)
meta = df[['genus', 'species']].to_dict('records')
np.save(f'{OS_ROOT}/ref_emb_logo.npy', emb)
json.dump(meta, open(f'{OS_ROOT}/metadata.json', 'w'))
print(f'embed selesai: {emb.shape} -> {OS_ROOT}/ref_emb_logo.npy', flush=True)

In [ ]:
# Sel 7 - Open-set AUROC atas genus UNSEEN + figur + snippet markdown utk paper 3.5(ii)
import numpy as np, json, csv, datetime
from sklearn.metrics import roc_auc_score
import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt

genera = np.array([d['genus'] for d in meta])
uniq = sorted(set(genera)); gid = {g: i for i, g in enumerate(uniq)}
gi = np.array([gid[g] for g in genera]); n = len(emb)
held_ids = {gid[g] for g in HELD_OUT if g in gid}

# top-K tetangga (chunked), self di-exclude
K = 64; idx = np.empty((n, K), 'int32'); sim = np.empty((n, K), 'float32')
for s in range(0, n, 2000):
    e = min(s+2000, n); block = emb[s:e] @ emb.T
    for r in range(e-s): block[r, s+r] = -np.inf
    part = np.argpartition(-block, K, axis=1)[:, :K]; rows = np.arange(e-s)[:, None]
    order = np.argsort(-block[rows, part], axis=1)
    idx[s:e] = part[rows, order]; sim[s:e] = block[rows, part][rows, order]
nbr_g = gi[idx]
known = ~np.isin(nbr_g, list(held_ids))           # tetangga = genus yang DIKENAL (bukan held-out)
first = known.argmax(axis=1)
dist = 1.0 - np.where(known.any(1), sim[np.arange(n), first], -1.0)

is_unk = np.isin(gi, list(held_ids)); known_dist = dist[~is_unk]
rows = []
for g in HELD_OUT:
    if g not in gid: continue
    gd = dist[gi == gid[g]]
    y = np.r_[np.ones(len(gd)), np.zeros(len(known_dist))]; sc = np.r_[gd, known_dist]
    rows.append((g, len(gd), round(roc_auc_score(y, sc), 4)))
pooled = roc_auc_score(is_unk.astype(int), dist)
aur = np.array([r[2] for r in rows])
print(f'Leave-{len(rows)}-genera-out (UNSEEN) open-set:')
print(f'  per-genus mean AUROC = {aur.mean():.3f} | median = {np.median(aur):.3f} | '
      f'range {aur.min():.3f}-{aur.max():.3f} | pooled = {pooled:.3f}')

with open(f'{OS_ROOT}/openset_logo_auroc.csv', 'w', newline='') as f:
    w = csv.writer(f); w.writerow(['genus', 'n', 'auroc']); w.writerows(rows)

# Figur 2-panel: (a) AUROC per genus, (b) contoh histogram jarak (Paphiopedilum)
EX = 'Paphiopedilum' if any(r[0] == 'Paphiopedilum' for r in rows) else \
     min(rows, key=lambda r: abs(r[2] - np.median(aur)))[0]
ex_auroc = next(r[2] for r in rows if r[0] == EX)
du = dist[gi == gid[EX]]
order = sorted(rows, key=lambda r: r[2])

fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 5))
a1.barh([r[0] for r in order], [r[2] for r in order], color='#2c6fbb')
a1.axvline(aur.mean(), color='#d1495b', lw=2, label=f'mean = {aur.mean():.3f}')
a1.axvline(0.5, color='#999', ls=':', lw=1.5, label='chance')
a1.set_xlim(0, 1); a1.set_xlabel('Open-set AUROC (genus unseen in training)')
a1.set_title(f'(a) Per-genus AUROC  (pooled = {pooled:.3f})'); a1.legend(frameon=False)

bins = np.linspace(0, max(du.max(), known_dist.max())*1.02, 36)
a2.hist(known_dist, bins=bins, color='#2c6fbb', alpha=0.7, density=True,
        label=f'in-distribution (known, n={len(known_dist):,})')
a2.hist(du, bins=bins, color='#d1495b', alpha=0.7, density=True,
        label=f'{EX} held-out (n={len(du)})')
a2.set_xlabel('Cosine distance to nearest known embedding'); a2.set_ylabel('Density')
a2.set_title(f'(b) Distance to known set: {EX}  (AUROC = {ex_auroc:.2f})')
a2.legend(frameon=False, fontsize=9)
fig.suptitle(f'Leave-{len(rows)}-genera-out open-set detection (DINOv2; genera unseen in training)',
             fontsize=13)
fig.tight_layout()
fig.savefig(f'{OS_ROOT}/fig_openset_logo.png', dpi=300, bbox_inches='tight')
fig.savefig(f'{OS_ROOT}/fig_openset_logo.pdf', bbox_inches='tight')
print('saved fig (2 panel) + csv ->', OS_ROOT)

date = datetime.date.today().isoformat()
print('\n----- tempel ke paper 3.5(ii) -----')
print(f'Leave-{len(rows)}-genera-out retraining ({date}): with {len(rows)} genera withheld from '
      f'training, DINOv2 detected them as novel by nearest-reference distance with mean per-genus '
      f'AUROC {aur.mean():.3f} (median {np.median(aur):.3f}, range {aur.min():.2f}-{aur.max():.2f}; '
      f'pooled {pooled:.3f}) - lower than the fixed-model bound of 0.963, as expected for '
      f'genera entirely unseen during training.')